# Panel Dataset of Post-1990 Democratizers with Inequality Measures
## Description
This notebook demonstrates the dataset created for analyzing how inequality, education, and democratic quality co-evolve across post-1990 democratizers.
**Dataset contents:**
- 5,804 country-year observations from 11 post-1990 democratizers (1990-2023)
- V-Dem v.14 democracy indices (v2x_libdem, v2pepwrsoc)
- Income inequality Gini coefficients (World Bank PIP)
- Education spending as %GDP (World Bank EdStats)
- Transition year dummies
**Research question:** Traces how inequality, education, and democratic-quality co-evolve across post-1990 democratizers, identifies what sustains resilience versus backsliding, and tests whether welfare-state institutions mediate the link.
**Source:** Data merged from OWID panels: V-Dem, World Bank PIP, LIED, OECD SOCX, Barro-Lee education, World Bank EdStats.


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab
_pip('seaborn')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'seaborn==0.13.2')


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import urllib.request

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print("Imports complete.")


In [ ]:
# GitHub URL for the demo data
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-27ddf1-inequality-political-equality-and-democr/main/round-2/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load data from GitHub URL with local fallback."""
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if Path("mini_demo_data.json").exists():
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

print("Data loading helper defined.")


In [ ]:
# Load the dataset
data = load_data()

# Explore the structure
print("Dataset structure:")
print(f"  Keys: {list(data.keys())}")
print(f"  Number of dataset groups: {len(data['datasets'])}")
print(f"  Dataset name: {data['datasets'][0]['dataset']}")
print(f"  Number of examples: {len(data['datasets'][0]['examples'])}")


## Exploring the Data Structure
Each example in the dataset has:
- `input`: JSON string of input features (inequality, education, transition dummies)
- `output`: Democratic quality score (V-Dem liberal democracy index)
- `metadata_*`: Additional metadata (country, year, fold assignment, etc.)


In [ ]:
# Examine first example
first_example = data['datasets'][0]['examples'][0]

print("Example structure:")
for key, value in first_example.items():
    val_str = str(value)
    if len(val_str) > 100:
        val_str = val_str[:100] + '...'
    print(f"  {key}: {val_str}")


## Creating Analysis DataFrame
Parse the input JSON strings and create a proper DataFrame for analysis.


In [ ]:
# Parse all examples into a DataFrame
examples = data['datasets'][0]['examples']

rows = []
for ex in examples:
    row = json.loads(ex['input'])  # Parse input features
    row['democratic_quality'] = float(ex['output'])  # Add target
    row['country'] = ex['metadata_country']
    row['year'] = ex['metadata_year']
    row['fold'] = ex['metadata_fold']
    rows.append(row)

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())


## Summary Statistics
Basic statistics for the key variables in our dataset.


In [ ]:
# Summary statistics
print("Summary Statistics:")
print("=" * 50)

numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col not in ['fold', 'year']:
        print(f"\n{col}:")
        print(f"  Mean: {df[col].mean():.4f}")
        print(f"  Std:  {df[col].std():.4f}")
        print(f"  Min:  {df[col].min():.4f}")
        print(f"  Max:  {df[col].max():.4f}")
        print(f"  Non-null: {df[col].notna().sum()}")


In [ ]:
# Country distribution
print("Observations per country:")
print(df['country'].value_counts().sort_index())


## Visualization: Democratic Quality Over Time
For the demo data (3 examples from Benin 1990-1992), we can see the early years of democratization.


In [ ]:
# Plot democratic quality over time
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Group by country and plot
for country in df['country'].unique():
    country_data = df[df['country'] == country].sort_values('year')
    ax.plot(country_data['year'], country_data['democratic_quality'],
            marker='o', label=country, linewidth=2)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Democratic Quality (V-Dem libdem)', fontsize=12)
ax.set_title('Democratic Quality Over Time (Demo Data)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Feature Exploration
Exploring the relationship between input features and democratic quality.


In [ ]:
# Check which features are available
feature_cols = [col for col in df.columns if col not in
                ['democratic_quality', 'country', 'year', 'fold']]
print("Available features:")
for col in feature_cols:
    non_null = df[col].notna().sum()
    print(f"  {col}: {non_null} non-null values")


## Full Dataset Information
The full dataset contains 5,804 observations. Here's what you would find:
- **Countries:** Benin, Bulgaria, Cape Verde, Czech Republic, Estonia, Ghana, Latvia, Lithuania, Malawi, Mongolia, Namibia, Panama, Romania, Slovakia, Slovenia, South Africa, Suriname, Chile, Brazil (20 democratizers)
- **Years:** 1990-2023 (34 years per country)
- **Key variables:**
  - `v2x_libdem`: V-Dem Liberal Democracy Index (target)
  - `v2pepwrsoc`: V-Dem Political Equality Index
  - `gini_income`: Income inequality Gini coefficient
  - `education_spending_gdp`: Education spending as % of GDP
  - `post_transition`: Dummy for post-transition years
  - `transition_year`: Year of democratic transition
## Next Steps for Analysis
With the full dataset, you could:
1. Run panel regression models (fixed effects, random effects)
2. Test whether inequality predicts democratic backsliding
3. Examine if education spending mediates the inequality-democracy link
4. Compare resilient vs. backsliding democracies
